# How To: Merge Fragmented Detections

When a single colony is split into multiple objects (fragmentation),
use refiners to merge the fragments back together.

In [ ]:
import phenotypic as pht
from phenotypic.data import load_yeast_plate
from phenotypic.enhance import GaussianBlur, CLAHE
from phenotypic.detect import OtsuDetector
from phenotypic.refine import NearestNeighborMerger, MaskFill, MaskCloser

In [ ]:
plate = load_yeast_plate()
plate = GaussianBlur(sigma=2.0)(plate)
plate = CLAHE(clip_limit=0.01)(plate)
plate = OtsuDetector()(plate)
print(f"Before merging: {plate.objmap.num_objects} objects")
plate.dash(overlay=True)

## Morphological Closing

`MaskCloser` bridges small gaps between nearby fragments using dilation
followed by erosion.

In [ ]:
closed = MaskCloser(width=5)(plate.copy())
closed = MaskFill()(closed)
print(f"After closing + fill: {closed.objmap.num_objects} objects")
closed.dash(overlay=True)

## Nearest-Neighbor Merging

`NearestNeighborMerger` merges objects that are within a specified
distance of each other, assigning smaller fragments to their nearest
larger neighbor.

In [ ]:
merged = NearestNeighborMerger()(plate.copy())
print(f"After NN merge: {merged.objmap.num_objects} objects")
merged.dash(overlay=True)

Use `MaskCloser` for small gaps between fragments, and
`NearestNeighborMerger` when fragments are further apart but clearly
belong to the same colony.